# **I. Tiền xử lý chung**

**1. Kiểm tra dữ liệu gốc**

In [ ]:
# ---Nạp thư viện---
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr

# ---Xác định thư mục dự án và nạp mô đun tiền xử lý---
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

project_path = str(PROJECT_ROOT.resolve())
if project_path not in sys.path:
    sys.path.insert(0, project_path)

import src.preprocessing as preprocessing

# ---Đọc bảng ứng dụng và tạo bản sao để xử lý---
apps_path = PROJECT_ROOT / "data/raw/googleplaystore.csv"
apps_raw = pd.read_csv(apps_path)
apps = apps_raw.copy()

print(f"Initial rows: {len(apps):,}")

**2. Kiểm tra và loại bản ghi lệch cột**  

Nhóm loại bản ghi có dấu hiệu lệch cột vì nhiều giá trị không phù hợp với ý nghĩa của thuộc tính và chưa có căn cứ khôi phục chính xác. Sau đó, nhóm loại các dòng trùng lặp hoàn toàn, giữ lần xuất hiện đầu tiên. Các bản ghi chỉ trùng tên ứng dụng được giữ lại để xem xét theo từng câu hỏi phân tích.

In [ ]:
# ---Hiển thị bản ghi có dấu hiệu lệch cột---
malformed_mask = preprocessing.malformed_app_mask(apps)
print("Suspected malformed rows:")
display(apps.loc[malformed_mask])

# ---Loại bản ghi lệch cột và báo cáo số dòng---
rows_before = len(apps)
apps = apps.loc[~malformed_mask].copy()

print(f"Rows before: {rows_before:,}")
print(f"Rows removed: {rows_before - len(apps):,}")
print(f"Rows after: {len(apps):,}")

Nhóm xác định và loại một bản ghi của ứng dụng Life Made WI-Fi Touchscreen Photo Frame, có nhiều giá trị không phù hợp với ý nghĩa của cột. Số dòng giảm từ 10.841 xuống 10.840.

**3. Loại bản sao hoàn toàn**  



In [ ]:
# ---Loại bản sao hoàn toàn và giữ lần xuất hiện đầu tiên---
rows_before = len(apps)
duplicate_count = apps.duplicated().sum()
apps = apps.drop_duplicates(keep="first").copy()

print(f"Rows before: {rows_before:,}")
print(f"Exact duplicates removed: {duplicate_count:,}")
print(f"Rows after: {len(apps):,}")
print(f"Remaining exact duplicates: {apps.duplicated().sum():,}")

Nhóm loại 483 bản sao giống nhau hoàn toàn, giữ lại bản ghi xuất hiện đầu tiên. Bảng còn 10.357 dòng và không còn dòng trùng lặp hoàn toàn.

**4. Kiểm tra các bản ghi trùng tên còn lại**

In [ ]:
# ---Thống kê các tên ứng dụng còn xuất hiện nhiều lần---
repeated_names = apps["App"].duplicated(keep=False)
extra_rows = apps["App"].duplicated().sum()
repeated_app_count = apps.loc[repeated_names, "App"].nunique()

name_summary = pd.Series({
    "Remaining rows": len(apps),
    "Distinct app names": apps["App"].nunique(),
    "Extra rows with repeated app names": extra_rows,
    "App names appearing more than once": repeated_app_count
})
display(name_summary)

Bảng còn 9.659 tên ứng dụng phân biệt. Trong đó, 523 tên xuất hiện nhiều lần, tạo ra 698 bản ghi thêm ngoài lần xuất hiện đầu tiên. Nhóm chưa loại các bản ghi này vì cùng tên không đồng nghĩa với thông tin hoàn toàn giống nhau; cách xử lý sẽ được xác định theo đơn vị phân tích của từng câu hỏi.

**5. Chuyển đổi các cột số**  

Nhóm sẽ thực hiện chuyển đổi các cột số như sau:  
- Giữ nguyên cột `Rating` vì đã có kiểu float64
- `Reviews` và `Installs` sang kiểu Int64
- `Price` sang float64  
- Giữ cột `Size` gốc dưới dạng `string`, thêm `Size_MB` kiểu `float64` để tính toán và `Size_varies` kiểu `boolean` để nhận diện placeholder  

Trước khi chuyển đổi, nhóm loại dấu phẩy và dấu + trong `Installs`, cùng ký hiệu $ trong `Price`. Giá trị `Installs` sau chuyển đổi vẫn là cận dưới của lượt cài đặt.

In [ ]:
rows_before = len(apps)

# ---Chuyển Rating và Reviews sang kiểu số---
rating = pd.to_numeric(apps["Rating"], errors="raise")
rating = rating.astype("float64")
reviews = pd.to_numeric(apps["Reviews"], errors="raise")
reviews = reviews.astype("Int64")

# ---Bỏ ký hiệu trong Installs và giữ ý nghĩa cận dưới---
install_text = apps["Installs"].astype("string")
install_text = install_text.str.replace(",", "", regex=False)
install_text = install_text.str.replace("+", "", regex=False)
installs = pd.to_numeric(install_text, errors="raise")
installs = installs.astype("Int64")

# ---Bỏ ký hiệu đô la và chuyển Price sang kiểu số---
price_text = apps["Price"].astype("string")
price_text = price_text.str.replace("$", "", regex=False)
price = pd.to_numeric(price_text, errors="raise")
price = price.astype("float64")

# ---Phân biệt kích thước thiếu và kích thước tùy thiết bị---
size_text = apps["Size"].astype("string").str.strip()
size_varies = (size_text == "Varies with device").fillna(False)
size_missing = size_text.isna() | (size_text == "").fillna(False)
size_parts = size_text.str.extract(r"^(\d+(?:\.\d+)?)([Mk])$")
unexpected_size = (
    ~size_missing & ~size_varies & size_parts[0].isna()
)

if unexpected_size.any():
    display(apps.loc[unexpected_size, ["App", "Size"]])
    raise ValueError("Unrecognized Size values remain.")

# ---Đổi kích thước sang MB theo quy ước k chia 1024---
size_value = pd.to_numeric(size_parts[0], errors="raise")
size_factor = size_parts[1].map({"M": 1.0, "k": 1 / 1024})
size_mb = (size_value * size_factor).astype("float64")

# ---Cập nhật bảng sau khi các phép chuyển đổi thành công---
apps["Rating"] = rating
apps["Reviews"] = reviews
apps["Installs"] = installs
apps["Price"] = price
apps["Size_MB"] = size_mb
apps["Size_varies"] = size_varies

# ---Kiểm tra kiểu dữ liệu và miền giá trị của năm cột số---
numeric_cols = ["Rating", "Reviews", "Size_MB", "Installs", "Price"]
numeric_values = apps[numeric_cols]
conversion_summary = pd.DataFrame({
    "Dtype": numeric_values.dtypes.astype(str),
    "Missing count": numeric_values.isna().sum(),
    "Negative count": (numeric_values < 0).sum(),
    "Min": numeric_values.min(),
    "Max": numeric_values.max()
})

print(f"Rows before: {rows_before:,}")
print(f"Rows after: {len(apps):,}")
display(conversion_summary)

invalid_rating = (
    apps["Rating"].notna() & ~apps["Rating"].between(1, 5)
)
nonpositive_size = (apps["Size_MB"] <= 0).sum()
print(f"Ratings outside 1-5: {invalid_rating.sum():,}")
print(f"Missing original Size: {size_missing.sum():,}")
print(f"Size varies with device: {size_varies.sum():,}")
print(f"Nonpositive Size_MB: {nonpositive_size:,}")

preview_cols = [
    "App", "Rating", "Reviews", "Size", "Size_MB",
    "Size_varies", "Installs", "Price"
]
display(apps[preview_cols].head())

**Nhận xét:**  
- Các cột số đã có kiểu dữ liệu đúng như dự kiến. `Reviews`, `Installs` và `Price` không có giá trị thiếu hoặc âm   
- `Rating` còn 1.465 giá trị thiếu; các điểm quan sát được đều nằm trong thang 1–5  
- `Size_MB` có 1.526 giá trị NaN, tương ứng hoàn toàn với các bản ghi Varies with device. Cột `Size` gốc không thiếu, cờ `Size_varies` giúp nhận diện các trường hợp kích thước chưa xác định  
- Không phát hiện kích thước nhỏ hơn hoặc bằng 0. Bước chuyển đổi không loại thêm bản ghi  

**6. Chuyển đổi cột thời gian `Last Updated`**  

Nhóm chuyển kiểu dữ liệu cột `Last Updated` từ chuỗi sang kiểu `datetime64[ns]` để hỗ trợ so sánh và trích xuất thông tin ngày tháng. Cột này biểu diễn lần cập nhật cuối của ứng dụng, không phải ngày phát hành hoặc ngày thu thập dữ liệu.

In [ ]:
rows_before = len(apps)
missing_before = apps["Last Updated"].isna().sum()

# ---Chuyển ngày theo định dạng gốc, ví dụ January 7, 2018---
last_updated = pd.to_datetime(
    apps["Last Updated"], format="%B %d, %Y", errors="raise"
)
apps["Last Updated"] = last_updated.astype("datetime64[ns]")

# ---Kiểm tra kiểu dữ liệu, giá trị thiếu và khoảng thời gian---
print(f"Rows before: {rows_before:,}")
print(f"Rows after: {len(apps):,}")
print(f"Dtype: {apps['Last Updated'].dtype}")
print(f"Missing values before: {missing_before:,}")
print(f"Missing values after: {apps['Last Updated'].isna().sum():,}")
print(f"Earliest date: {apps['Last Updated'].min():%d/%m/%Y}")
print(f"Latest date: {apps['Last Updated'].max():%d/%m/%Y}")
display(apps[["App", "Last Updated"]].head())

Cột `Last Updated` đã được chuyển sang kiểu `datetime64[ns]`, không phát sinh giá trị thiếu. Ngày cập nhật nhỏ nhất là 21/05/2010, lớn nhất là 08/08/2018. Số dòng giữ nguyên ở 10.357.

**7. Xử lý `Type` thiếu và kiểm tra với `Price`**

In [ ]:
rows_before = len(apps)

# ---Nhận diện và hiển thị các bản ghi thiếu Type---
type_text = apps["Type"].astype("string").str.strip()
missing_type = type_text.isna() | (type_text == "").fillna(False)
print("Rows with missing Type before processing:")
display(apps.loc[missing_type, ["App", "Type", "Price", "Installs"]])

# ---Suy ra Type còn thiếu từ giá hợp lệ---
infer_free = missing_type & (apps["Price"] == 0)
infer_paid = missing_type & (apps["Price"] > 0)
apps.loc[infer_free, "Type"] = "Free"
apps.loc[infer_paid, "Type"] = "Paid"

# ---Kiểm tra nhãn thiếu, nhãn lạ và mâu thuẫn với Price---
type_after = apps["Type"].astype("string").str.strip()
missing_after = (
    type_after.isna() | (type_after == "").fillna(False)
)
invalid_type = ~missing_after & ~type_after.isin(["Free", "Paid"])
conflict = (
    ((type_after == "Free") & (apps["Price"] > 0))
    | ((type_after == "Paid") & (apps["Price"] == 0))
).fillna(False)

type_summary = pd.Series({
    "Missing Type before": missing_type.sum(),
    "Inferred Free": infer_free.sum(),
    "Inferred Paid": infer_paid.sum(),
    "Missing Type after": missing_after.sum(),
    "Invalid Type labels": invalid_type.sum(),
    "Type-Price conflicts": conflict.sum()
})
display(type_summary)

print("Rows with invalid Type or Type-Price conflicts:")
display(apps.loc[invalid_type | conflict, ["App", "Type", "Price"]])
print(f"Rows before: {rows_before:,}")
print(f"Rows after: {len(apps):,}")

Bản ghi "Command & Conquer: Rivals" thiếu `Type` và có `Price` = 0, nên nhóm suy ra nhãn Free. Sau xử lý, cột `Type` không còn giá trị thiếu hoặc nhãn không hợp lệ. Nhóm cũng không phát hiện thêm bản ghi có `Type` mâu thuẫn với `Price`. Do đó số dòng được giữ nguyên.

Các giá trị thiếu của `Rating`, `Current Ver` và `Android Ver` được giữ trong bảng chung. Việc lọc theo điểm đánh giá hoặc tạo nhóm `Unknown` sẽ được thực hiện riêng khi câu hỏi cần đến.

**8. Chuyển kiểu các cột phân loại và văn bản**  

Ở phần data exploration, nhóm có một số đề xuất về việc chuyển đổi các cột biểu diễn nhãn sang kiểu `category` và các cột văn bản sang kiểu `string`. Mỗi tổ hợp trong `Genres` tiếp tục được giữ như một nhãn riêng. Cụ thể:    
- `Category`, `Type`, `Content Rating`, `Genres` → `category`
- `App`, `Size`, `Current Ver`, `Android Ver` → `string`

Giữ nguyên các nhãn tổ hợp, kể cả `Education;Education`; chưa gộp nhãn hiếm hoặc chuẩn hóa các thành phần trong bước tiền xử lý chung.

In [ ]:
rows_before = len(apps)
categorical_cols = ["Category", "Type", "Content Rating", "Genres"]
string_cols = ["App", "Size", "Current Ver", "Android Ver"]
converted_cols = categorical_cols + string_cols

# ---Ghi nhận trạng thái trước khi chuyển kiểu---
dtype_before = apps[converted_cols].dtypes.astype(str)
missing_before = apps[converted_cols].isna().sum()
distinct_before = apps[converted_cols].nunique()

# ---Chuyển các cột phân loại và văn bản---
for col in categorical_cols:
    apps[col] = apps[col].astype("category")

for col in string_cols:
    apps[col] = apps[col].astype("string")

# ---Đối chiếu kiểu dữ liệu, số giá trị thiếu và số nhãn---
conversion_summary = pd.DataFrame({
    "Dtype before": dtype_before,
    "Dtype after": apps[converted_cols].dtypes.astype(str),
    "Missing before": missing_before,
    "Missing after": apps[converted_cols].isna().sum(),
    "Distinct before": distinct_before,
    "Distinct after": apps[converted_cols].nunique()
})
display(conversion_summary)
print(f"Rows before: {rows_before:,}")
print(f"Rows after: {len(apps):,}")

**9. Kiểm tra lại cấu trúc dữ liệu của các cột**

In [ ]:
# ---Đặt lại chỉ số liên tục và xem cấu trúc bảng---
apps = apps.reset_index(drop=True)
apps.info()

Sau tiền xử lý, bảng có 10.357 dòng và 15 cột, gồm 13 cột gốc và hai cột mới `Size_MB`, `Size_varies`. Các cột số, phân loại, văn bản và ngày tháng đã có kiểu dữ liệu phù hợp với kế hoạch

# **II. Đặt câu hỏi và phân tích**

Mỗi thành viên phụ trách hai câu hỏi. Mỗi câu hỏi không dùng ML cần đáp ứng ít nhất hai tiêu chí chiều sâu; các câu hỏi của nhóm cùng bao phủ ít nhất bốn tiêu chí. Các tiêu chí gồm kết hợp dữ liệu, tái cấu trúc dữ liệu hoặc phân tích thời gian, tạo biến dẫn xuất, kiểm soát yếu tố gây nhiễu, lượng hóa độ bất định và kiểm tra độ bền của kết quả.

### **2.1. So sánh điểm đánh giá giữa ứng dụng miễn phí và trả phí**

**Người thực hiện:** Phạm Hoàng Tuấn Anh

**Câu hỏi:** Trong các nhóm ứng dụng có cùng danh mục và khoảng cài đặt, điểm đánh giá giữa ứng dụng miễn phí và trả phí khác nhau như thế nào? Kết luận có thay đổi khi nâng số lượt đánh giá tối thiểu từ 10 lên 100 không?

**Động lực:** Kết quả khám phá cho thấy hai nhóm có điểm trung bình khác nhau, nhưng cũng khác nhau về danh mục và quy mô cài đặt. Vì vậy, cần so sánh trong các nhóm tương đồng thay vì chỉ dựa vào trung bình toàn mẫu. Kết quả có thể giúp nhà phát triển tham khảo đặc điểm điểm đánh giá của hai mô hình thu phí khi cân nhắc cách cung cấp sản phẩm. Điểm đánh giá không đo đầy đủ chất lượng hoặc mức độ hài lòng; phân tích này không xác định tác động nhân quả của việc thu phí.

**Các tiêu chí chiều sâu:**

- **D3:** Tạo biến phân nhóm cận dưới lượt cài đặt.
- **D4:** So sánh trong từng tổ hợp danh mục và nhóm cài đặt, sau đó tổng hợp bằng cùng một bộ trọng số.
- **D5:** Ước lượng khoảng tin cậy 95% bằng lấy mẫu lặp theo tên ứng dụng.
- **D6:** Kiểm tra kết quả ở hai ngưỡng lượt đánh giá và khi tạm loại ứng dụng có giá từ 100 USD trở lên.

#### **1. Chuẩn bị dữ liệu và xác định phạm vi so sánh**

Nhóm sử dụng bảng `apps` đã làm sạch ở mục I và thực hiện các bước sau:

1. Chọn một bản ghi cho mỗi cặp `App`–`Category`: ưu tiên số lượt đánh giá lớn nhất, sau đó ngày cập nhật gần hơn, cuối cùng là thứ tự dòng gốc. Quy tắc này không chứng minh bản ghi được thu thập gần nhất. Việc giữ từng cặp giúp tránh mất danh mục của ứng dụng có nhiều danh mục.
2. Giữ điểm trong thang 1–5 và loại miễn phí hoặc trả phí. Từ cùng bảng này, tạo hai mẫu có ít nhất 10 và 100 lượt đánh giá. Không điền điểm thiếu.
3. Chia cận dưới lượt cài đặt thành năm khoảng: dưới 1.000; từ 1.000 đến dưới 10.000; từ 10.000 đến dưới 100.000; từ 100.000 đến dưới 1 triệu; từ 1 triệu trở lên.
4. Chỉ so sánh các tổ hợp danh mục–khoảng cài đặt có ít nhất 10 ứng dụng ở mỗi loại tại **cả hai ngưỡng đánh giá**. Ngưỡng này nhằm sàng lọc nhóm nhỏ, không bảo đảm ước lượng chính xác. Giữ cùng các tổ hợp để tránh thay đổi phạm vi nhóm khi kiểm tra độ bền.

Mỗi dòng phân tích đại diện cho một cặp ứng dụng–danh mục. Tên ứng dụng chỉ là dấu hiệu nhận diện thay thế vì dữ liệu không có mã ứng dụng; các cặp cùng tên có thể phụ thuộc nhau. Các bước chọn và loại dòng đều được báo cáo riêng.

In [ ]:
# ---Tạo hàm ghi nhận số dòng qua từng bước---
q1_audit_rows = []


def q1_record_step(label, before, after):
    q1_audit_rows.append({
        "Step": label,
        "Rows before": before,
        "Rows removed": before - after,
        "Rows after": after
    })


# ---Chọn một bản ghi đại diện cho mỗi cặp ứng dụng và danh mục---
q1_units = apps.sort_values(
    ["Reviews", "Last Updated"],
    ascending=[False, False], kind="stable"
)
q1_units = q1_units.drop_duplicates(["App", "Category"]).copy()
q1_record_step("Select app-category pairs", len(apps), len(q1_units))

valid_rating = q1_units["Rating"].between(1, 5)
valid_type = q1_units["Type"].isin(["Free", "Paid"])
q1_valid = q1_units.loc[valid_rating & valid_type].copy()
q1_record_step(
    "Keep valid ratings and types", len(q1_units), len(q1_valid)
)

# ---Chia cận dưới lượt cài đặt thành năm khoảng---
q1_brackets = [
    "<1K", "1K-<10K", "10K-<100K", "100K-<1M", ">=1M"
]
q1_valid["Install_Bracket"] = pd.cut(
    q1_valid["Installs"].astype("float64"),
    bins=[0, 1_000, 10_000, 100_000, 1_000_000, np.inf],
    labels=q1_brackets, right=False
)
if q1_valid["Install_Bracket"].isna().any():
    raise ValueError("Some install values have no bracket.")
q1_record_step(
    "Create install brackets", len(q1_valid), len(q1_valid)
)

# ---Tạo hai mẫu theo ngưỡng số lượt đánh giá---
q1_samples = {}
for threshold in [10, 100]:
    sample = q1_valid.loc[q1_valid["Reviews"] >= threshold].copy()
    q1_samples[threshold] = sample
    q1_record_step(
        f"Keep at least {threshold} reviews",
        len(q1_valid), len(sample)
    )

# ---Đếm số ứng dụng theo danh mục, khoảng cài đặt và loại---
q1_group_cols = ["Category", "Install_Bracket"]
q1_counts = {}
for threshold, sample in q1_samples.items():
    counts = sample.groupby(
        q1_group_cols + ["Type"], observed=True
    ).size()
    counts = counts.unstack("Type", fill_value=0)
    q1_counts[threshold] = counts.reindex(
        columns=["Free", "Paid"], fill_value=0
    )

# ---Chọn tổ hợp có ít nhất mười ứng dụng mỗi loại ở cả hai mẫu---
q1_min_per_type = 10
q1_coverage = q1_counts[10].add_suffix("_10").join(
    q1_counts[100].add_suffix("_100"), how="left"
)
q1_coverage = q1_coverage.fillna(0).astype(int)
q1_coverage["Eligible"] = (
    q1_coverage >= q1_min_per_type
).all(axis=1)
q1_common_groups = q1_coverage.index[q1_coverage["Eligible"]]
if len(q1_common_groups) == 0:
    raise ValueError("No groups are eligible at both thresholds.")

q1_analysis = {}
for threshold, sample in q1_samples.items():
    keys = pd.MultiIndex.from_frame(sample[q1_group_cols])
    selected = sample.loc[keys.isin(q1_common_groups)].copy()
    q1_analysis[threshold] = selected
    q1_record_step(
        f"Keep common groups at threshold {threshold}",
        len(sample), len(selected)
    )

# ---Giữ cùng bộ trọng số lấy từ mẫu có ít nhất mười đánh giá---
q1_reference_counts = q1_analysis[10].groupby(
    q1_group_cols, observed=True
).size()
q1_reference_counts = q1_reference_counts.reindex(q1_common_groups)
q1_weights = q1_reference_counts / q1_reference_counts.sum()

# ---Hiển thị số dòng và mức bao phủ của hai mẫu---
q1_audit = pd.DataFrame(q1_audit_rows)
display(q1_audit)
print(f"Observed groups at threshold 10: {len(q1_coverage):,}")
print(f"Common eligible groups: {len(q1_common_groups):,}")
display(q1_coverage)

q1_sample_rows = []
for threshold in [10, 100]:
    before = q1_samples[threshold]
    after = q1_analysis[threshold]
    q1_sample_rows.append({
        "Review threshold": threshold,
        "Pairs before group selection": len(before),
        "Pairs compared": len(after),
        "Retained percent": len(after) / len(before) * 100,
        "Distinct app names": after["App"].nunique(),
        "Free": (after["Type"] == "Free").sum(),
        "Paid": (after["Type"] == "Paid").sum()
    })
q1_sample_summary = pd.DataFrame(q1_sample_rows)
display(q1_sample_summary.round(2))

**Nhận xét phạm vi dữ liệu:**

- Từ 10.357 dòng, nhóm chọn được 9.744 cặp ứng dụng–danh mục; sau khi chọn điểm và loại ứng dụng hợp lệ còn 8.280 cặp. Hai ngưỡng 10 và 100 lượt đánh giá lần lượt giữ lại 7.693 và 6.383 cặp trước khi xét nhóm đối chiếu.
- Trong 157 tổ hợp danh mục–nhóm cài đặt quan sát ở ngưỡng 10, chỉ có **8 tổ hợp thuộc 5 danh mục** đủ điều kiện tại cả hai ngưỡng.
- Mẫu so sánh chính ở ngưỡng 10 có **1.462 cặp**, gồm 1.262 miễn phí và 200 trả phí, bằng **19,00%** số cặp đạt ngưỡng. Ở ngưỡng 100, mẫu có **1.137 cặp**, gồm 957 miễn phí và 180 trả phí, bằng **17,81%** số cặp đạt ngưỡng.
- Trong từng mẫu cuối, mỗi tên ứng dụng chỉ xuất hiện một lần. Quy trình lấy mẫu theo tên vẫn được giữ để xử lý nhất quán nếu phạm vi nhóm thay đổi.
- Phạm vi đối chiếu khá hẹp. Kết quả chỉ mô tả những tổ hợp đủ điều kiện, không đại diện cho toàn bộ các ứng dụng có điểm đánh giá. Nâng ngưỡng làm thay đổi thành phần ứng dụng ngay cả khi giữ nguyên danh sách tổ hợp.

#### **2. Ước lượng chênh lệch và độ bất định**

Trong mỗi tổ hợp danh mục–nhóm cài đặt, nhóm tính điểm trung bình của ứng dụng trả phí trừ điểm trung bình của ứng dụng miễn phí. Chênh lệch sau phân nhóm là trung bình có trọng số của các hiệu số này. Trọng số lấy từ số cặp ứng dụng–danh mục ở ngưỡng 10 và giữ cố định cho cả hai ngưỡng, để hai loại ứng dụng được so sánh theo cùng cơ cấu nhóm.

Bảng kết quả còn trình bày chênh lệch toàn mẫu và chênh lệch chưa phân nhóm trên cùng phạm vi đối chiếu. Các số này phục vụ mô tả, không thay thế kết quả đã phân nhóm.

Khoảng tin cậy 95% được tính bằng 1.000 lần lấy mẫu có hoàn lại theo **tên ứng dụng**. Mọi dòng cùng tên nhận cùng số lần xuất hiện trong mỗi lần lấy mẫu. Cùng một lần lấy mẫu được dùng cho hai ngưỡng để ước lượng thêm độ thay đổi của chênh lệch. Các tổ hợp và trọng số được giữ cố định, nên khoảng tin cậy có điều kiện theo phạm vi đã chọn. Không sử dụng kiểm định hai mẫu độc lập cho các dòng có thể thuộc cùng một ứng dụng.

In [ ]:
# ---Tính trung bình từng loại trong mỗi tổ hợp---
def q1_group_means(frame, app_counts=None):
    data = frame[q1_group_cols + ["Type", "Rating"]].copy()
    data["Count"] = 1.0
    if app_counts is not None:
        data["Count"] = frame["App"].map(app_counts)
    data["Rating_sum"] = data["Rating"] * data["Count"]

    totals = data.groupby(
        q1_group_cols + ["Type"], observed=True
    )[["Count", "Rating_sum"]].sum()
    counts = totals["Count"].unstack("Type", fill_value=0)
    sums = totals["Rating_sum"].unstack("Type", fill_value=0)
    counts = counts.reindex(
        index=q1_common_groups,
        columns=["Free", "Paid"], fill_value=0
    )
    sums = sums.reindex(
        index=q1_common_groups,
        columns=["Free", "Paid"], fill_value=0
    )
    means = sums / counts.replace(0, np.nan)
    return means, counts


# ---Tính chênh lệch có trọng số và chênh lệch chưa phân nhóm---
def q1_adjusted_difference(frame):
    means, counts = q1_group_means(frame)
    if (counts == 0).any().any():
        return np.nan
    differences = means["Paid"] - means["Free"]
    return (q1_weights * differences).sum()


def q1_raw_difference(frame):
    means = frame.groupby("Type", observed=True)["Rating"].mean()
    return means.get("Paid", np.nan) - means.get("Free", np.nan)


# ---Lấy mẫu theo tên để các dòng cùng ứng dụng đi cùng nhau---
q1_app_names = q1_analysis[10]["App"].drop_duplicates().to_numpy()
q1_n_apps = len(q1_app_names)
q1_rng = np.random.default_rng(42)
q1_boot_draws = []
q1_attempts = 0

while len(q1_boot_draws) < 1_000 and q1_attempts < 5_000:
    q1_attempts += 1
    sampled_indices = q1_rng.integers(q1_n_apps, size=q1_n_apps)
    sampled_names = q1_app_names[sampled_indices]
    app_counts = pd.Series(sampled_names).value_counts()
    app_counts = app_counts.reindex(q1_app_names, fill_value=0)

    differences = []
    for threshold in [10, 100]:
        means, counts = q1_group_means(
            q1_analysis[threshold], app_counts
        )
        if (counts == 0).any().any():
            differences.append(np.nan)
        else:
            group_difference = means["Paid"] - means["Free"]
            difference = (q1_weights * group_difference).sum()
            differences.append(difference)

    if np.isfinite(differences).all():
        q1_boot_draws.append(differences)

if len(q1_boot_draws) < 1_000:
    raise ValueError("Too few valid bootstrap samples.")
q1_boot_draws = np.array(q1_boot_draws)

# ---Tổng hợp ước lượng và khoảng tin cậy ở từng ngưỡng---
q1_stats_rows = []
for position, threshold in enumerate([10, 100]):
    frame = q1_analysis[threshold]
    means, counts = q1_group_means(frame)
    low, high = np.percentile(
        q1_boot_draws[:, position], [2.5, 97.5]
    )
    q1_stats_rows.append({
        "Review threshold": threshold,
        "Full sample gap": q1_raw_difference(q1_samples[threshold]),
        "Common sample gap": q1_raw_difference(frame),
        "Weighted Free rating": (q1_weights * means["Free"]).sum(),
        "Weighted Paid rating": (q1_weights * means["Paid"]).sum(),
        "Adjusted gap": q1_adjusted_difference(frame),
        "CI lower": low,
        "CI upper": high
    })
q1_statistics = pd.DataFrame(q1_stats_rows)
display(q1_statistics.round(4))

# ---Dùng cùng lần lấy mẫu để tính thay đổi giữa hai ngưỡng---
q1_change = (
    q1_statistics.loc[1, "Adjusted gap"]
    - q1_statistics.loc[0, "Adjusted gap"]
)
q1_boot_changes = q1_boot_draws[:, 1] - q1_boot_draws[:, 0]
q1_change_ci = np.percentile(q1_boot_changes, [2.5, 97.5])
low, high = q1_change_ci
invalid_draws = q1_attempts - len(q1_boot_draws)
print(f"Gap change from threshold 10 to 100: {q1_change:+.4f}")
print(f"95% CI for the change: [{low:.4f}; {high:.4f}]")
print(f"Valid bootstrap samples: {len(q1_boot_draws):,}")
print(f"Samples with missing groups: {invalid_draws:,}")

**Nhận xét thống kê:**

- Với ít nhất 10 lượt đánh giá, điểm trung bình theo cơ cấu chung là **4,087** ở nhóm miễn phí và **4,364** ở nhóm trả phí. Chênh lệch sau phân nhóm là **+0,277 điểm**, với khoảng tin cậy 95% **[0,228; 0,329]**.
- Với ít nhất 100 lượt đánh giá, chênh lệch tương ứng là **+0,275 điểm**, với khoảng tin cậy 95% **[0,214; 0,340]**. Cả hai khoảng đều nằm trên 0 theo phương pháp lấy mẫu đã chọn.
- Cần phân biệt thay đổi phạm vi và thay đổi cách tổng hợp. Ở ngưỡng 10, chênh lệch toàn mẫu là +0,096 điểm, còn chênh lệch chưa phân nhóm trên phạm vi chung là +0,271 điểm. So với +0,277 điểm sau phân nhóm, sự khác biệt lớn với toàn mẫu chủ yếu gắn với phạm vi đối chiếu khác nhau; không được quy toàn bộ thay đổi cho việc kiểm soát danh mục và mức cài đặt.

#### **3. Trực quan hóa kết quả sau phân nhóm**

Biểu đồ thứ nhất trình bày chênh lệch tổng hợp đã phân nhóm và khoảng tin cậy ở hai ngưỡng lượt đánh giá. Biểu đồ thứ hai trình bày chênh lệch theo từng danh mục thuộc phạm vi chung; trong mỗi danh mục, trọng số các nhóm cài đặt cũng được giữ cố định giữa hai ngưỡng. Biểu đồ theo danh mục chỉ mô tả các ước lượng điểm, không thể hiện ý nghĩa thống kê của từng danh mục.

In [ ]:
# ---Vẽ chênh lệch tổng hợp cùng khoảng tin cậy---
fig, ax = plt.subplots(figsize=(9, 4.5))
positions = np.arange(len(q1_statistics))
ax.hlines(
    positions, q1_statistics["CI lower"],
    q1_statistics["CI upper"], color="#2166ac", linewidth=2
)
ax.scatter(
    q1_statistics["Adjusted gap"], positions,
    color="#2166ac", zorder=3
)
ax.axvline(0, color="black", linestyle="--", linewidth=1)
ax.set_yticks(positions)
ax.set_yticklabels(["At least 10 reviews", "At least 100 reviews"])
ax.set_xlabel("Paid minus Free rating (1-5 scale)")
ax.set_title("Adjusted rating gaps with 95% bootstrap CIs")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

# ---Tổng hợp chênh lệch theo danh mục với cùng cơ cấu cài đặt---
q1_group_differences = pd.DataFrame(index=q1_common_groups)
for threshold in [10, 100]:
    means, counts = q1_group_means(q1_analysis[threshold])
    q1_group_differences[f"Threshold {threshold}"] = (
        means["Paid"] - means["Free"]
    )
q1_group_differences["Weight"] = q1_weights

q1_category_rows = []
for category, frame in q1_group_differences.groupby(
    level="Category", observed=True
):
    weights = frame["Weight"] / frame["Weight"].sum()
    q1_category_rows.append({
        "Category": category,
        "Threshold 10": (weights * frame["Threshold 10"]).sum(),
        "Threshold 100": (weights * frame["Threshold 100"]).sum(),
        "Groups": len(frame)
    })
q1_category_results = pd.DataFrame(q1_category_rows)
q1_category_results = q1_category_results.sort_values("Threshold 10")
display(q1_category_results.round(4))

# ---Vẽ chênh lệch theo danh mục ở hai ngưỡng đánh giá---
height = max(4.5, len(q1_category_results) * 0.5)
fig, ax = plt.subplots(figsize=(10, height))
positions = np.arange(len(q1_category_results))
plot_settings = [(-0.16, 10, "#2166ac"), (0.16, 100, "#b2182b")]
for offset, threshold, color in plot_settings:
    ax.barh(
        positions + offset,
        q1_category_results[f"Threshold {threshold}"],
        height=0.3, color=color,
        label=f"At least {threshold} reviews"
    )
ax.set_yticks(positions)
ax.set_yticklabels(q1_category_results["Category"])
ax.axvline(0, color="black", linestyle="--", linewidth=1)
ax.set_xlabel("Paid minus Free rating (1-5 scale)")
ax.set_title("Rating gaps by category after install adjustment")
ax.legend()
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

**Nhận xét biểu đồ:**

- Biểu đồ thứ nhất cho thấy hai ước lượng chênh lệch tổng hợp gần nhau; cả hai khoảng tin cậy đều nằm phía trên đường mốc 0. Độ thay đổi giữa hai ngưỡng được đánh giá riêng ở phần kiểm tra độ bền, không suy ra chỉ từ việc hai khoảng tin cậy chồng lấn.
- Trong năm danh mục được giữ lại, các ước lượng điểm đều dương ở cả hai ngưỡng. Tuy nhiên, mức chênh lệch khác nhau giữa danh mục: `TOOLS` có hiệu số lớn hơn các danh mục còn lại, còn `PERSONALIZATION` có hiệu số nhỏ hơn.
- Mỗi danh mục chỉ được đại diện bởi các khoảng cài đặt đủ điều kiện, không phải toàn bộ ứng dụng trong danh mục. Biểu đồ thứ hai không có khoảng tin cậy riêng cho danh mục nên không dùng để kết luận ý nghĩa thống kê hoặc xếp hạng chắc chắn giữa các danh mục.

#### **4. Kiểm tra độ bền của kết quả**

Nhóm đối chiếu kết quả ở ngưỡng 10 và 100 lượt đánh giá trên cùng các tổ hợp và cùng trọng số. Ngoài ra, nhóm tính lại chênh lệch sau khi tạm loại bản ghi có giá từ 100 USD trở lên. Số bản ghi bị loại được tính riêng trong từng mẫu, không sử dụng số 20 của bảng gốc.

Ngưỡng giá cao là lựa chọn kiểm tra độ nhạy, không có nghĩa các bản ghi này là sai hoặc không có chất lượng. Nếu sau khi lọc có tổ hợp thiếu một loại ứng dụng, chênh lệch sẽ không được tính thay vì thay giá trị thiếu bằng 0.

In [ ]:
# ---Tạm loại giá cao và tính lại chênh lệch với cùng trọng số---
q1_price_rows = []
for threshold in [10, 100]:
    before = q1_analysis[threshold]
    after = before.loc[before["Price"] < 100].copy()
    means, counts = q1_group_means(after)
    small_groups = (counts < 10).any(axis=1).sum()
    missing_groups = (counts == 0).any(axis=1).sum()
    q1_price_rows.append({
        "Review threshold": threshold,
        "Rows before": len(before),
        "High-price rows removed": len(before) - len(after),
        "Rows after": len(after),
        "Original gap": q1_adjusted_difference(before),
        "Gap without high prices": q1_adjusted_difference(after),
        "Groups below 10 in either type": small_groups,
        "Groups missing a type": missing_groups
    })
q1_price_sensitivity = pd.DataFrame(q1_price_rows)
q1_price_sensitivity["Gap change"] = (
    q1_price_sensitivity["Gap without high prices"]
    - q1_price_sensitivity["Original gap"]
)
display(q1_price_sensitivity.round(4))

**Nhận xét kiểm tra độ bền:**

- Khi nâng ngưỡng từ 10 lên 100 lượt đánh giá, chênh lệch sau phân nhóm thay đổi từ **+0,277** xuống **+0,275 điểm**. Độ thay đổi là **-0,002 điểm**, với khoảng tin cậy 95% **[-0,043; 0,043]** chứa 0. Trong phạm vi và phương pháp hiện tại, chưa có bằng chứng rõ về sự thay đổi của hiệu số khi nâng ngưỡng.
- Khi tạm loại giá từ 100 USD trở lên, **mỗi mẫu loại 3 bản ghi**, không phải 20 bản ghi của bảng gốc. Chênh lệch sau phân nhóm lần lượt là **+0,281** và **+0,280 điểm**, tăng khoảng **0,0042** và **0,0046 điểm** so với ban đầu. Không có tổ hợp nào giảm xuống dưới 10 ứng dụng ở một loại.
- Chiều chênh lệch không đảo trong hai lựa chọn đã kiểm tra. Điều này hỗ trợ tính ổn định của kết quả trong mẫu đối chiếu, không chứng minh một quy luật chung hoặc việc ứng dụng trả phí có chất lượng tốt hơn.

#### **5. Trả lời câu hỏi và giới hạn**

**Trả lời câu hỏi:** Trong 8 tổ hợp danh mục–khoảng cài đặt đủ điều kiện, nhóm trả phí có điểm trung bình cao hơn nhóm miễn phí khoảng **0,277 điểm** ở ngưỡng 10 lượt đánh giá và **0,275 điểm** ở ngưỡng 100. Đây là các chênh lệch được tổng hợp bằng cùng cơ cấu nhóm. Hai khoảng tin cậy 95% đều nằm trên 0.

**Độ bền:** Nâng ngưỡng không đảo chiều kết quả và chỉ làm hiệu số thay đổi khoảng -0,002 điểm; khoảng tin cậy của thay đổi chứa 0. Tạm loại giá cao cũng không đảo chiều kết quả. Các kết luận về thay đổi giữa hai ngưỡng dựa trên hiệu số sau phân nhóm, không dựa vào mức tăng của chênh lệch toàn mẫu.

**Ý nghĩa thực tiễn:** Kết quả cho thấy mối liên hệ giữa mô hình thu phí và điểm đánh giá trong phạm vi đối chiếu. Đây là một thông tin tham khảo cho nhà phát triển, chưa đủ để khuyến nghị chuyển sang thu phí hoặc khẳng định mua ứng dụng trả phí sẽ đem lại trải nghiệm tốt hơn.

**Giới hạn:**


- Kết quả chỉ áp dụng cho các tổ hợp có đủ ứng dụng miễn phí và trả phí ở cả hai ngưỡng. Các ứng dụng thiếu điểm hoặc không có nhóm đối chiếu không nằm trong phạm vi kết luận.
- Khoảng cài đặt dựa trên cận dưới, không phải lượt cài chính xác. Phân nhóm không loại hết khác biệt về mức độ phổ biến và các yếu tố chưa quan sát.
- Một tên có thể không phải một mã ứng dụng duy nhất. Quy tắc chọn bản ghi đại diện và lấy mẫu theo tên chỉ là cách xử lý theo thông tin sẵn có.
- Trên bảng trước khi chọn tổ hợp, ứng dụng có nhiều danh mục có thể đóng góp nhiều cặp. Trong mẫu đối chiếu hiện tại mỗi tên chỉ còn một cặp; trọng số tổng hợp vẫn được xác định theo tổ hợp, không phải điểm trung bình toàn mẫu thông thường.
- Khoảng tin cậy mô tả độ bất định trong mẫu và phạm vi đã chọn; không khắc phục sai lệch thu thập hoặc khiến dataset đại diện cho toàn bộ Google Play.
- Dữ liệu không đo quảng cáo, giao dịch mua trong ứng dụng hoặc chất lượng trải nghiệm. Không suy ra nguyên nhân của điểm số, đánh giá ảo hay hiệu quả của việc chuyển mô hình thu phí.

### **2.2. Độ lệch giữa Rating hiển thị và cảm xúc của review**

**Người thực hiện:** *(điền tên thành viên)*

**Câu hỏi:** Khi kết hợp bảng ứng dụng và bảng review, điểm `Rating` tích lũy có nhất quán với cảm xúc trung bình (`Sentiment_Polarity`) và tỷ lệ review `Negative` hay không, sau khi kiểm soát theo `Category` và quy mô `Installs`?

**Động lực:** `Rating` là con số tích lũy cả vòng đời, còn review là mẫu văn bản nhỏ của từng ứng dụng. Nhận diện nhóm ứng dụng có `Rating` > 4,2 nhưng tỷ lệ review tiêu cực cao giúp Product Manager chú ý sớm tới rủi ro chất lượng. Bảng review không có thời gian nên phân tích chỉ đo **độ lệch**, không kết luận nguyên nhân (ví dụ lỗi phiên bản mới).

**Chiều sâu:** D1 (gộp bảng), D3 (biến dẫn xuất), D4 (kiểm soát nhiễu), D5 (bootstrap, p-value) và D6 (kiểm tra độ bền).

#### **1. Làm sạch bảng review và gộp với bảng apps (D1)**

1. `preprocessing.preprocess_reviews` loại review thiếu nội dung rồi loại bản sao hoàn toàn, có báo cáo số dòng từng bước.
2. Tổng hợp review về cấp ứng dụng (mục 2 bên dưới), chọn một bản ghi cho mỗi tên ứng dụng trong bảng `apps` (ưu tiên `Reviews` lớn nhất, rồi `Last Updated` mới hơn).
3. Gộp inner join với `validate="one_to_one"`, báo cáo số ứng dụng khớp và không khớp.

In [ ]:
# ---Đọc và làm sạch bảng đánh giá bằng mô đun dùng chung---
reviews_path = (
    PROJECT_ROOT / "data/raw/googleplaystore_user_reviews.csv"
)
reviews_raw = pd.read_csv(reviews_path)
reviews, reviews_report = preprocessing.preprocess_reviews(
    reviews_raw
)
display(reviews_report)

Từ 64.295 dòng, nhóm loại 26.868 dòng thiếu nội dung và 7.735 bản sao hoàn toàn, còn **29.692** dòng review.

#### **2. Tổng hợp về cấp ứng dụng và gộp bảng (D1, D3)**

Biến dẫn xuất từ review: `Mean_Polarity`, `Std_Polarity` (độ phân cực), `Negative_Review_Share` (số review `Negative` chia tổng review của ứng dụng), `N_Reviews`. Biến `Negative_Share_PN` bỏ review `Neutral` khỏi mẫu số, chỉ dùng cho kiểm tra độ bền (D6).

In [ ]:
# ---Đếm nhãn cảm xúc và tổng hợp đánh giá về cấp ứng dụng---
q2_counts = reviews.groupby(["App", "Sentiment"]).size()
q2_counts = q2_counts.unstack("Sentiment", fill_value=0)
q2_reviews = reviews.groupby("App").agg(
    N_Reviews=("Sentiment", "size"),
    Mean_Polarity=("Sentiment_Polarity", "mean"),
    Std_Polarity=("Sentiment_Polarity", "std")
)
q2_reviews["Negative_Review_Share"] = (
    q2_counts["Negative"] / q2_reviews["N_Reviews"]
)

# ---Tạo tỷ lệ chỉ tính trên đánh giá tích cực và tiêu cực---
positive_negative = q2_counts["Negative"] + q2_counts["Positive"]
q2_reviews["Negative_Share_PN"] = (
    q2_counts["Negative"] / positive_negative
)
q2_reviews = q2_reviews.reset_index()

# ---Chọn một bản ghi đại diện cho mỗi tên ứng dụng---
q2_apps = apps.sort_values(
    ["Reviews", "Last Updated"], ascending=False, kind="stable"
)
q2_apps = q2_apps.drop_duplicates("App")
q2_apps = q2_apps[["App", "Category", "Rating", "Installs"]]

# ---Gộp hai bảng và báo cáo số ứng dụng khớp tên---
q2_unmatched = sorted(set(q2_reviews["App"]) - set(q2_apps["App"]))
q2 = q2_reviews.merge(
    q2_apps, on="App", how="inner", validate="one_to_one"
)
q2["Category"] = q2["Category"].astype(str)
merge_summary = pd.Series({
    "Clean review rows": len(reviews),
    "Apps in review table": len(q2_reviews),
    "Distinct apps in app table": len(q2_apps),
    "Matched apps": len(q2),
    "Review apps missing from app table": len(q2_unmatched),
    "Matched apps with missing Rating": q2["Rating"].isna().sum()
}, name="Count")
display(merge_summary)
print("Examples of unmatched app names:", q2_unmatched[:5])

Bảng review có **865** ứng dụng, trong đó **816** khớp với bảng `apps` và **49** không có trong bảng `apps` (nhiều tên là ứng dụng tiếng Trung hoặc ghi khác tên). Phép gộp one-to-one, 816 ứng dụng đều có `Rating`.

#### **3. Chọn phạm vi và tạo biến lệch chuẩn hóa (D3)**

- Mỗi ứng dụng chỉ có trung bình 34 review, một số chỉ có 1 review, nên nhóm chỉ giữ ứng dụng có **≥ 10 review** làm phân tích chính (725 ứng dụng). Ngưỡng **≥ 30** (581 ứng dụng) dùng ở bước kiểm tra độ bền.
- `Z_Gap = Z(Rating) − Z(Mean_Polarity)`, Z-score tính trong mẫu đã lọc. `Z_Gap` dương nghĩa là `Rating` cao hơn mức cảm xúc review.
- `Install_Tier` chia `Installs` thành bốn tầng vì `Installs` chỉ là cận dưới.
- `At_Risk` là ứng dụng có `Rating` > 4,2 và `Negative_Review_Share` thuộc phân vị 75 trở lên của mẫu.

In [ ]:
# ---Khai báo ngưỡng đánh giá và các tầng cài đặt---
MIN_REVIEWS = 10
MIN_REVIEWS_CHECK = 30
Q2_TIERS = [0, 100_000, 1_000_000, 10_000_000, np.inf]
Q2_TIER_LABELS = ["<100K", "100K-<1M", "1M-<10M", ">=10M"]


# ---Lọc mẫu và tạo độ lệch chuẩn hóa cùng cờ rủi ro---
def q2_prepare(min_reviews):
    enough_reviews = q2["N_Reviews"] >= min_reviews
    has_rating = q2["Rating"].notna()
    data = q2.loc[enough_reviews & has_rating].copy()

    rating_z = (
        (data["Rating"] - data["Rating"].mean())
        / data["Rating"].std(ddof=0)
    )
    polarity_z = (
        (data["Mean_Polarity"] - data["Mean_Polarity"].mean())
        / data["Mean_Polarity"].std(ddof=0)
    )
    data["Z_Gap"] = rating_z - polarity_z
    data["Install_Tier"] = pd.cut(
        data["Installs"].astype(float), Q2_TIERS,
        labels=Q2_TIER_LABELS, right=False
    )
    negative_cutoff = data["Negative_Review_Share"].quantile(0.75)
    data["At_Risk"] = (
        (data["Rating"] > 4.2)
        & (data["Negative_Review_Share"] >= negative_cutoff)
    )
    return data


# ---Tính tương quan hạng sau khi kiểm soát danh mục và cài đặt---
def q2_partial_rho(data, x, y):
    category_flags = pd.get_dummies(
        data["Category"], drop_first=True
    ).astype(float)
    controls = category_flags.copy()
    controls.insert(0, "Intercept", 1.0)
    controls["Log_installs"] = np.log10(
        data["Installs"].astype(float)
    )
    controls = controls.to_numpy()
    x_rank = data[x].rank().to_numpy()
    y_rank = data[y].rank().to_numpy()

    # ---Ước lượng hạng bằng bình phương tối thiểu---
    x_coefficients = np.linalg.lstsq(controls, x_rank, rcond=None)[0]
    y_coefficients = np.linalg.lstsq(controls, y_rank, rcond=None)[0]
    x_residual = x_rank - controls @ x_coefficients
    y_residual = y_rank - controls @ y_coefficients

    # ---Lấy tương quan Pearson của hai phần dư hạng---
    return np.corrcoef(x_residual, y_residual)[0, 1]


# ---Lấy mẫu lại ứng dụng để tính khoảng tin cậy 95 phần trăm---
def q2_boot_ci(
    data, x, y=None, method="spearman", n_boot=2000, seed=42
):
    rng = np.random.default_rng(seed)
    values = []
    for draw in range(n_boot):
        indices = rng.integers(0, len(data), len(data))
        sample = data.iloc[indices]
        if method == "mean":
            value = sample[x].mean()
        elif method == "partial":
            value = q2_partial_rho(sample, x, y)
        else:
            value = spearmanr(sample[x], sample[y])[0]
        values.append(value)
    return tuple(np.percentile(values, [2.5, 97.5]))


# ---Chuẩn bị mẫu chính và xem các biến dẫn xuất---
q2_main = q2_prepare(MIN_REVIEWS)
print(f"Review threshold: {MIN_REVIEWS}")
print(f"Apps: {len(q2_main):,}")
print(f"Categories: {q2_main['Category'].nunique()}")
summary_cols = [
    "Mean_Polarity", "Std_Polarity", "Negative_Review_Share", "Z_Gap"
]
display(q2_main[summary_cols].describe().round(3))
print("Apps with Rating > 4.2:", (q2_main["Rating"] > 4.2).sum())
print("At-risk apps:", q2_main["At_Risk"].sum())

#### **4. Tương quan tổng thể và sau khi kiểm soát (D4, D5)**

Nhóm dùng Spearman vì `Rating` lệch trái và `Installs` chỉ là thang thứ bậc. Hệ số riêng phần là Spearman của phần dư hạng sau khi loại ảnh hưởng của `Category` (biến giả) và `log10(Installs)`. Khoảng tin cậy 95% lấy từ bootstrap lấy mẫu lại ứng dụng.

In [ ]:
# ---So sánh tương quan tổng thể và tương quan riêng phần---
rows = []
for column in ["Mean_Polarity", "Negative_Review_Share"]:
    correlation, p_value = spearmanr(
        q2_main["Rating"], q2_main[column]
    )
    low, high = q2_boot_ci(q2_main, "Rating", column)
    partial = q2_partial_rho(q2_main, "Rating", column)
    partial_low, partial_high = q2_boot_ci(
        q2_main, "Rating", column, method="partial", n_boot=1000
    )
    rows.append({
        "Variable compared with Rating": column,
        "Spearman": correlation,
        "p-value": p_value,
        "CI lower": low,
        "CI upper": high,
        "Partial correlation": partial,
        "Partial CI": f"[{partial_low:.3f}; {partial_high:.3f}]"
    })
q2_overall = pd.DataFrame(rows)
display(q2_overall.round(4))

- Trên 725 ứng dụng, `Rating` và `Mean_Polarity` có tương quan dương vừa phải (Spearman **0,296**, CI [0,231; 0,363], p < 0,001). `Rating` và tỷ lệ `Negative` tương quan âm yếu hơn (**−0,231**, CI [−0,300; −0,166]).
- Sau khi kiểm soát `Category` và `Installs`, độ mạnh tăng lên **0,390** (CI [0,324; 0,456]) và **−0,317** (CI [−0,387; −0,252]). Vậy `Category` và quy mô cài đặt làm nhiễu nhẹ mối liên hệ, không tạo ra mối liên hệ.
- Mức nhất quán chỉ vừa phải: `Rating` giải thích được phần nhỏ biến thiên cảm xúc review.

#### **5. Kiểm soát theo danh mục và tầng cài đặt, độ bất định (D4, D5)**

Chỉ giữ nhóm có ít nhất 30 ứng dụng (11 danh mục). `SOCIAL` không đủ ứng dụng nên không nằm trong so sánh. Với mỗi nhóm, nhóm tính Spearman (kèm p-value), tỷ lệ `Negative` trung bình và `Z_Gap` trung bình, cùng CI bootstrap 95%. Do chia nhóm nhỏ nên CI khá rộng.

In [ ]:
# ---Thống kê từng nhóm có đủ số ứng dụng tối thiểu---
def q2_group_table(data, column, min_n):
    rows = []
    for name, group in data.groupby(column, observed=True):
        if len(group) < min_n:
            continue
        correlation, p_value = spearmanr(
            group["Rating"], group["Mean_Polarity"]
        )
        rho_ci = q2_boot_ci(
            group, "Rating", "Mean_Polarity", n_boot=1000
        )
        negative_ci = q2_boot_ci(
            group, "Negative_Review_Share",
            method="mean", n_boot=1000
        )
        gap_ci = q2_boot_ci(
            group, "Z_Gap", method="mean", n_boot=1000
        )
        rows.append({
            column: name,
            "n": len(group),
            "Spearman": correlation,
            "p-value": p_value,
            "CI Spearman": rho_ci,
            "Mean negative share": (
                group["Negative_Review_Share"].mean()
            ),
            "CI Negative": negative_ci,
            "Mean Z_Gap": group["Z_Gap"].mean(),
            "CI Z_Gap": gap_ci
        })
    return pd.DataFrame(rows)


# ---Định dạng khoảng tin cậy bằng vòng lặp dễ đọc---
def q2_format_table(table):
    result = table.copy()
    for column in result.columns:
        if column.startswith("CI"):
            intervals = []
            for low, high in result[column]:
                intervals.append(f"[{low:.3f}; {high:.3f}]")
            result[column] = intervals
    return result.round(3)


q2_by_cat = q2_group_table(q2_main, "Category", 30)
q2_by_cat = q2_by_cat.sort_values("Spearman").reset_index(drop=True)
q2_by_tier = q2_group_table(q2_main, "Install_Tier", 30)
display(q2_format_table(q2_by_cat))
display(q2_format_table(q2_by_tier))

- **Khác biệt rõ giữa các danh mục:** Spearman cao nhất ở `MEDICAL` (0,664), `SPORTS` (0,586), `TRAVEL_AND_LOCAL` (0,541), `FINANCE` (0,486). `GAME` chỉ **0,095** (CI [−0,172; 0,344], p = 0,421), nghĩa là trong `GAME` `Rating` gần như không dự đoán được cảm xúc review.
- **`GAME` là danh mục lệch nhất:** tỷ lệ `Negative` trung bình **35,1%** (CI [32,0%; 38,0%]), cao nhất, và `Z_Gap` trung bình **+1,33** (CI [1,10; 1,55]). `Rating` của game cao hơn nhiều so với cảm xúc review. `FAMILY` cũng lệch dương (+0,60). `MEDICAL` lệch âm (−0,71).
- **Theo tầng cài đặt:** Spearman giảm từ khoảng 0,38–0,43 ở các tầng nhỏ xuống **0,248** ở tầng ≥10M. Tỷ lệ `Negative` tăng dần theo quy mô, từ 11,1% (<100K) lên 25,1% (≥10M). `Z_Gap` đổi dấu từ âm sang +0,45 ở tầng ≥10M. Ứng dụng càng lớn thì `Rating` càng cao hơn so với cảm xúc review.

#### **6. Trực quan hóa**

In [ ]:
# ---Vẽ điểm đánh giá và cảm xúc theo từng tầng cài đặt---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for tier, group in q2_main.groupby("Install_Tier", observed=True):
    axes[0].scatter(
        group["Rating"], group["Mean_Polarity"],
        s=18, alpha=0.6, label=tier
    )
risk = q2_main.loc[q2_main["At_Risk"]]
axes[0].scatter(
    risk["Rating"], risk["Mean_Polarity"], s=60,
    facecolors="none", edgecolors="red",
    label="Rating > 4.2 and high negative share"
)
axes[0].axvline(4.2, color="gray", linestyle="--", linewidth=1)
axes[0].set_title("Displayed rating and review sentiment")
axes[0].set_xlabel("Rating (stars)")
axes[0].set_ylabel("Mean polarity (-1 to 1)")
axes[0].legend(title="Install floor", fontsize=8)

# ---Vẽ tương quan theo danh mục và khoảng tin cậy---
table = q2_by_cat.reset_index(drop=True)
ci_lower = []
ci_upper = []
category_labels = []
for i in range(len(table)):
    row = table.iloc[i]
    ci_lower.append(row["CI Spearman"][0])
    ci_upper.append(row["CI Spearman"][1])
    category_labels.append(f"{row['Category']} (n={row['n']})")
positions = np.arange(len(table))
axes[1].hlines(positions, ci_lower, ci_upper, color="C0")
axes[1].scatter(table["Spearman"], positions, color="C0")
axes[1].axvline(0, color="gray", linestyle="--", linewidth=1)
axes[1].set_yticks(positions)
axes[1].set_yticklabels(category_labels)
axes[1].set_title("Rating-polarity correlation by category")
axes[1].set_xlabel("Spearman correlation with 95% bootstrap CI")
plt.tight_layout()
plt.show()

Biểu đồ trái cho thấy các ứng dụng At_Risk (khoanh đỏ) nằm ở vùng `Rating` cao nhưng cảm xúc trung bình thấp, tập trung ở tầng cài đặt lớn. Biểu đồ phải cho thấy hầu hết CI của Spearman ở `GAME`, `DATING`, `PERSONALIZATION`, `HEALTH_AND_FITNESS` và `FAMILY` chạm hoặc chứa 0, trong khi `MEDICAL`, `SPORTS`, `TRAVEL_AND_LOCAL`, `FINANCE`, `PRODUCTIVITY` và `TOOLS` nằm hẳn về phía dương.

In [ ]:
# ---Vẽ tỷ lệ đánh giá tiêu cực theo danh mục---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
table = q2_by_cat.sort_values("Mean negative share")
positions = np.arange(len(table))
ci_lower = []
ci_upper = []
for low, high in table["CI Negative"]:
    ci_lower.append(low)
    ci_upper.append(high)
axes[0].hlines(positions, ci_lower, ci_upper, color="C0")
axes[0].scatter(table["Mean negative share"], positions, color="C0")
axes[0].set_yticks(positions)
axes[0].set_yticklabels(table["Category"])
axes[0].set_title("Mean negative review share by category")
axes[0].set_xlabel("Negative review share with 95% bootstrap CI")

# ---Vẽ độ lệch chuẩn hóa theo danh mục---
table = q2_by_cat.sort_values("Mean Z_Gap")
positions = np.arange(len(table))
ci_lower = []
ci_upper = []
for low, high in table["CI Z_Gap"]:
    ci_lower.append(low)
    ci_upper.append(high)
axes[1].hlines(positions, ci_lower, ci_upper, color="C1")
axes[1].scatter(table["Mean Z_Gap"], positions, color="C1")
axes[1].axvline(0, color="gray", linestyle="--", linewidth=1)
axes[1].set_yticks(positions)
axes[1].set_yticklabels(table["Category"])
axes[1].set_title("Standardized rating-sentiment gap by category")
axes[1].set_xlabel(
    "Mean Z_Gap with 95% bootstrap CI\n"
    ">0: rating exceeds review sentiment on standardized scales"
)
plt.tight_layout()
plt.show()

# ---Hiển thị ví dụ có rủi ro và thống kê theo tầng cài đặt---
risk_cols = [
    "App", "Category", "Rating", "N_Reviews", "Mean_Polarity",
    "Negative_Review_Share", "Z_Gap"
]
risk_examples = q2_main.loc[q2_main["At_Risk"], risk_cols]
risk_examples = risk_examples.sort_values(
    "Negative_Review_Share", ascending=False
)
display(risk_examples.head(10).round(3))
print("At-risk apps by install tier:")
risk_by_tier = q2_main.groupby(
    "Install_Tier", observed=True
)["At_Risk"].agg(["sum", "mean"])
display(risk_by_tier.round(3))

- Biểu đồ trái: `GAME` có tỷ lệ `Negative` cao nhất và CI không giao với các danh mục khác. Biểu đồ phải: `Z_Gap` của `GAME` và `FAMILY` dương, của `MEDICAL`, `DATING` và `PERSONALIZATION` âm (CI không chứa 0).
- Có **85** ứng dụng At_Risk. **62** trong số đó (23,2% tầng ≥10M) thuộc tầng ≥10M, tầng <100K không có ứng dụng nào. Danh sách đầu bảng chủ yếu là game phổ biến (Candy Crush, Angry Birds, Cooking Fever, Gardenscapes).

#### **7. Kiểm tra độ bền (D6)**

Nhóm lặp lại phân tích với hai lựa chọn hợp lý khác: (a) ngưỡng review 10 hoặc 30, (b) tỷ lệ `Negative` tính trên tất cả review hoặc chỉ trên review `Positive` và `Negative`. Với mỗi biến thể, nhóm báo cáo Spearman, CI bootstrap và hệ số riêng phần.

In [ ]:
# ---Đổi ngưỡng đánh giá và mẫu số của tỷ lệ tiêu cực---
rows = []
share_definitions = [
    ("Negative_Review_Share", "All reviews"),
    ("Negative_Share_PN", "Positive + Negative")
]
for threshold in [MIN_REVIEWS, MIN_REVIEWS_CHECK]:
    data = q2_prepare(threshold)
    for share_column, label in share_definitions:
        sample = data.dropna(subset=[share_column])
        row = {
            "Review threshold": threshold,
            "Negative share denominator": label,
            "n": len(sample)
        }
        comparisons = [
            ("Mean_Polarity", "Polarity"),
            (share_column, "Negative share")
        ]
        for column, name in comparisons:
            low, high = q2_boot_ci(
                sample, "Rating", column, n_boot=1000
            )
            row[f"Spearman {name}"] = spearmanr(
                sample["Rating"], sample[column]
            )[0]
            row[f"CI {name}"] = f"[{low:.3f}; {high:.3f}]"
            row[f"Partial {name}"] = q2_partial_rho(
                sample, "Rating", column
            )
        rows.append(row)
q2_robust = pd.DataFrame(rows)
with pd.option_context("display.max_columns", None):
    display(q2_robust.round(3))

- Kết luận không đổi ở cả bốn biến thể. Spearman `Rating`–`Mean_Polarity` nằm trong khoảng **0,296–0,297** (riêng phần 0,390–0,416). Spearman `Rating`–`Negative share` luôn âm, từ **−0,231** đến **−0,306** (riêng phần −0,317 đến −0,404).
- Bỏ `Neutral` khỏi mẫu số làm mối liên hệ âm mạnh hơn một chút (−0,295 so với −0,231 ở ngưỡng 10), nhưng chiều và mức độ vẫn cùng mức. CI của các biến thể chồng lấn nhau, nên không có bằng chứng rõ rằng lựa chọn làm sạch làm đổi kết luận.
- `Mean_Polarity` không phụ thuộc cách tính `Negative`, nên hai dòng cùng ngưỡng có cùng Spearman `Polarity`.

#### **8. Trả lời câu hỏi và giới hạn**

**Trả lời:** `Rating` hiển thị phản ánh cảm xúc review ở mức vừa phải và không đồng đều. Trên 725 ứng dụng, Spearman với `Mean_Polarity` là 0,296 và với tỷ lệ `Negative` là −0,231; sau khi kiểm soát `Category` và `Installs` lần lượt là 0,390 và −0,317. Kết quả ổn định qua hai ngưỡng review và hai cách tính `Negative`. Độ lệch tập trung ở nhóm cụ thể: `GAME` có Spearman 0,095 (không có ý nghĩa), tỷ lệ `Negative` 35,1% và `Z_Gap` +1,33. Tầng ≥10M cũng lệch dương (+0,45) và chứa 62 trong 85 ứng dụng At_Risk. Product Manager nên xem `Rating` cao của game và ứng dụng cài đặt lớn kèm tỷ lệ review tiêu cực.

**Giới hạn:**
- Bảng review chỉ là mẫu khoảng 30–40 review cho mỗi ứng dụng, không phải toàn bộ review, và không có thời gian nên không kiểm chứng được giả thuyết "review phản ánh phiên bản gần đây".
- `Sentiment_Polarity` do mô hình gán tự động, không phải điểm của người dùng, và chỉ 816 ứng dụng khớp (49 ứng dụng review không có trong bảng `apps`).
- Mỗi danh mục chỉ có 30–74 ứng dụng nên CI rộng; ngưỡng At_Risk (`Rating` > 4,2, phân vị 75 của `Negative`) là lựa chọn của nhóm.
- Đây là mối liên hệ quan sát, không phải quan hệ nhân quả.